In [ ]:
!pip install -q torch torchvision torchaudio wandb

In [ ]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
import wandb
wandb.login(WANDB_API_KEY)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet34
import time, tracemalloc

def make_cifar_resnet(arch='resnet18'):
    if arch == 'resnet18':
        model = torchvision.models.resnet18(weights=None)
    elif arch == 'resnet34':
        model = torchvision.models.resnet34(weights=None)
    # CIFAR adaptation
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 10)
    return model

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform_test)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                           shuffle=True, num_workers=2)
testloader  = torch.utils.data.DataLoader(testset, batch_size=128,
                                           shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
def train_model(model, epochs=100, run_name='teacher-resnet34'):
    wandb.init(project='cnn-compression', name=run_name,
               config={'arch': run_name, 'epochs': epochs, 'lr': 0.1})

    optimizer = torch.optim.SGD(model.parameters(), lr=0.1,
                                 momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        train_acc = 100. * correct / total

        # Eval
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)
        test_acc = 100. * correct / total

        wandb.log({'epoch': epoch+1, 'train_acc': train_acc,
                   'test_acc': test_acc, 'lr': scheduler.get_last_lr()[0]})

        if (epoch+1) % 20 == 0:
            print(f"[{epoch+1}/{epochs}] Train: {train_acc:.2f}% | Test: {test_acc:.2f}%")

    wandb.finish()
    return model

teacher = make_cifar_resnet('resnet34')
teacher = train_model(teacher, epochs=200, run_name='teacher-resnet34')
torch.save(teacher.state_dict(), 'teacher_resnet34_cifar10.pth')
print("Teacher saved.")

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels,
                      T=4.0, alpha=0.1):
    """
    alpha  = weight on hard-label CE loss
    (1-alpha) = weight on soft-target KL loss
    T²     = temperature scaling factor (restores gradient magnitude)
    """
    # Soft target loss
    soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_teacher = F.softmax(teacher_logits / T, dim=1)
    kl_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (T ** 2)

    # Hard target loss
    ce_loss = F.cross_entropy(student_logits, labels)

    return alpha * ce_loss + (1 - alpha) * kl_loss

In [ ]:
def train_with_distillation(teacher, student, epochs=100,
                             T=4.0, alpha=0.1, run_name='distilled-resnet18'):
    wandb.init(project='cnn-compression', name=run_name,
               config={'arch': 'resnet18-distilled', 'teacher': 'resnet34',
                       'T': T, 'alpha': alpha, 'epochs': epochs})

    teacher = teacher.to(device)
    teacher.eval()  # Teacher frozen — no gradients needed
    for p in teacher.parameters():
        p.requires_grad = False

    student = student.to(device)
    optimizer = torch.optim.SGD(student.parameters(), lr=0.1,
                                 momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0
    for epoch in range(epochs):
        student.train()
        running_loss, correct, total = 0, 0, 0

        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            with torch.no_grad():
                teacher_logits = teacher(inputs)

            optimizer.zero_grad()
            student_logits = student(inputs)

            loss = distillation_loss(student_logits, teacher_logits,
                                      labels, T=T, alpha=alpha)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = student_logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        train_acc = 100. * correct / total

        # Eval
        student.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = student(inputs)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)
        test_acc = 100. * correct / total

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(student.state_dict(), 'distilled_resnet18_best.pth')

        wandb.log({'epoch': epoch+1, 'train_acc': train_acc,
                   'test_acc': test_acc, 'loss': running_loss/len(trainloader),
                   'lr': scheduler.get_last_lr()[0]})

        if (epoch+1) % 20 == 0:
            print(f"[{epoch+1}/{epochs}] Train: {train_acc:.2f}% | Test: {test_acc:.2f}% | Best: {best_acc:.2f}%")

    print(f"\nDistillation complete. Best test accuracy: {best_acc:.2f}%")
    wandb.finish()
    return student

# Load teacher if you saved it
teacher = make_cifar_resnet('resnet34')
teacher.load_state_dict(torch.load('teacher_resnet34_cifar10.pth'))

student = make_cifar_resnet('resnet18')  # Fresh init, NOT your baseline weights
student = train_with_distillation(teacher, student, epochs=200, T=4.0, alpha=0.1)

In [ ]:
def benchmark(model, label='distilled-resnet18'):
    model_cpu = model.cpu().eval()

    # Latency (100-run avg, single sample)
    dummy = torch.randn(1, 3, 32, 32)
    for _ in range(20):  # warmup
        _ = model_cpu(dummy)

    times = []
    with torch.no_grad():
        for _ in range(100):
            t0 = time.perf_counter()
            _ = model_cpu(dummy)
            times.append((time.perf_counter() - t0) * 1000)
    avg_latency = sum(times) / len(times)

    # RAM
    tracemalloc.start()
    with torch.no_grad():
        _ = model_cpu(dummy)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # Params + size
    params = sum(p.numel() for p in model_cpu.parameters()) / 1e6
    torch.save(model_cpu.state_dict(), f'{label}_temp.pth')
    import os
    size_mb = os.path.getsize(f'{label}_temp.pth') / 1e6
    os.remove(f'{label}_temp.pth')

    print(f"\n── {label} ──")
    print(f"  Params:   {params:.2f}M")
    print(f"  Size:     {size_mb:.2f} MB")
    print(f"  Latency:  {avg_latency:.2f} ms")
    print(f"  Peak RAM: {peak/1e6:.2f} MB")
    return {'params': params, 'size_mb': size_mb,
            'latency_ms': avg_latency, 'peak_ram_mb': peak/1e6}

student.load_state_dict(torch.load('distilled_resnet18_best.pth'))
metrics = benchmark(student, 'distilled-resnet18')